# Use case — `Utility/LensNN.py`

Score every component pair with the trained ResTCN lens classifier. The inference code reconstructs the architecture and feature channels saved at training time.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility.LensNN import run_lensnn_inference

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"        # Preprocessed canonical light curves.
MODEL_PATH = PROJECT_ROOT / "models" / "resTCN.pt"                    # Exported ResTCN checkpoint and configuration.
OUTPUT_CSV = PROJECT_ROOT / "results" / "nn_pair_predictions.csv"      # One probability per available pair.

BATCH_SIZE = 64              # Pairs scored together; reduce if GPU/CPU memory is limited.
KEEP_FAILED_PAIRS = True     # Preserve unscorable pairs with Proba=NaN.
DEVICE = None                # None auto-selects CUDA, Apple MPS or CPU; explicit "cpu" is safest.
FILTER_OUTLIERS = True       # Exclude rows marked by flag_outlier when that column exists.
RETURN_ERRORS = True         # Also return a table explaining every failed pair.
VERBOSE = True               # Print device, input counts and final summary.

SOURCE_COLUMN = None         # None uses the checkpoint/default source_id column.
COMPONENT_COLUMN = None      # None uses lensComponentSourceId.
TIME_COLUMN = None           # None uses epoch_obs_jd.
FLUX_COLUMN = None           # None uses flux_obs.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Saved ResTCN model not found: {MODEL_PATH}")

In [ ]:
result = run_lensnn_inference(
    input_csv=INPUT_CSV,
    model_path=MODEL_PATH,
    output_csv=OUTPUT_CSV,
    batch_size=BATCH_SIZE,
    keep_failed_pairs=KEEP_FAILED_PAIRS,
    verbose=VERBOSE,
    device=DEVICE,
    source_col=SOURCE_COLUMN,
    component_col=COMPONENT_COLUMN,
    time_col=TIME_COLUMN,
    flux_col=FLUX_COLUMN,
    filter_outliers=FILTER_OUTLIERS,
    return_errors=RETURN_ERRORS,
)
predictions, errors = result if RETURN_ERRORS else (result, None)
display(predictions.sort_values("Proba", ascending=False).head(20))
if errors is not None and len(errors):
    display(errors.head())